# MCP-Bench: Tool DAG Similarity & Consistency Analysis

This notebook analyses the structural consistency of tool dependency graphs (DAGs) produced by LLM agents across repeated runs of the same tasks. The core question it answers is: **given the same prompt, how consistently does a model produce the same execution plan?**

The analysis is built in layers:
- **Sections 1–3** establish clean, validated data
- **Section 4** defines all reusable metric functions
- **Sections 5–7** perform the actual analyses
- **Section 8** produces all visualisations

Each analysis section depends only on the functions defined in Section 4 and the dataframes produced by earlier sections. Sections 7 and 8 contain no new graph-level computation — only aggregations and plots of data already built upstream.

---
## Section 1: Setup & Imports

### Rationale
All dependencies, configuration constants, and global plot settings live in this section. Keeping everything here means the notebook can be re-run cleanly from the top without hunting for scattered imports. It also makes the dependency surface explicit — anyone reading the notebook can see at a glance what libraries are required.

This section has no analytical content. It is purely infrastructural.

### 1.1 — Package installation

**Purpose:** Ensure all required packages are available in the environment.

A `pip install` cell using the `!` shell escape. Should install:
- `networkx` — graph construction and all DAG-level operations
- `numpy` — matrix operations on the pairwise similarity matrices
- `scipy` — hierarchical clustering (`linkage`, `fcluster`) and condensed distance matrix conversion (`squareform`)
- `pandas` — tabular storage of per-task, per-model result dataframes
- `matplotlib` — base plotting
- `seaborn` — heatmaps and styled statistical plots
- `scikit-learn` — MDS embedding for the 2D scatter visualisation

Mark this cell so it can be skipped if the environment is already set up (e.g. a comment `# run once`).

### 1.2 — Imports

**Purpose:** Load all libraries in a single cell so that re-running any later cell never fails due to a missing import.

Group imports logically:
1. Standard library (`json`, `pathlib`, `itertools`, `collections`)
2. Third-party data/graph (`networkx`, `numpy`, `pandas`, `scipy`)
3. Machine learning (`sklearn.manifold.MDS`)
4. Visualisation (`matplotlib`, `seaborn`)

Set a consistent random seed here for any stochastic operations (MDS, clustering) so results are reproducible across re-runs.

### 1.3 — Configuration constants

**Purpose:** Centralise all magic numbers and file paths so nothing is hardcoded deeper in the notebook.

Should define:
- `DATA_PATH` — path to the raw JSON/JSONL output file from MCP-Bench
- `GROUND_TRUTH_PATH` — path to the ground truth graph definitions
- `MODEL_LIST` — the explicit list of model identifiers present in the data
- `N_REPS` — number of repetitions per task × model combination
- `SIMILARITY_THRESHOLD` — the Jaccard cutoff below which two graphs are considered meaningfully different (used in outlier detection later)
- `CONSISTENCY_CLUSTER_DISTANCE` — the Ward linkage distance threshold used to count plan clusters per task

Making these constants explicit here means they can be tuned without touching any analysis cell.

### 1.4 — Plot style defaults

**Purpose:** Set global matplotlib/seaborn aesthetics once so all plots in the notebook share a consistent look.

Should set figure size defaults, DPI, colour palette, font sizes, and spine/grid visibility. Doing this here prevents each plotting cell from needing its own style boilerplate.

---
## Section 2: Data Loading & Graph Construction

### Rationale
Before any metric can be computed, raw JSON records must be converted into `networkx.DiGraph` objects — the data structure that all later functions operate on. This section is the only place in the notebook where files are read from disk. Everything downstream works with the in-memory graph objects produced here.

Separating loading from analysis means that if the raw data format changes, only this section needs updating.

The primary output is two nested dictionaries:
- `llm_graphs[(task_id, model, rep)]` → `nx.DiGraph`
- `ground_truth_graphs[task_id]` → `nx.DiGraph`

All keys are tuples so they can be easily grouped, filtered, and iterated.

### 2.1 — `load_raw_records(path)`

**Purpose:** Read the MCP-Bench output file and return a list of raw record dicts.

Handles:
- JSON array format (single `json.load`) and JSONL format (line-by-line `json.loads`)
- Raises a clear error if the file is missing or malformed, rather than failing silently later

Returns a flat list of dicts, each containing at minimum: `task_id`, `model`, `rep`, `llm_graph`, and `ground_truth_graph`.

### 2.2 — `build_dag(graph_record)`

**Purpose:** Convert a single raw graph record (a dict with `nodes` and `edges` fields) into a `networkx.DiGraph`.

Nodes should use the tool name string as the node identifier — this is what makes edge Jaccard work without any alignment step. If the record uses numeric IDs with a separate name field, this function resolves the mapping so that all downstream edges are `(tool_name_a, tool_name_b)` tuples.

Any node-level metadata (e.g. tool category, MCP server) should be stored as node attributes on the graph for potential use in later analyses.

### 2.3 — `build_all_graphs(records)`

**Purpose:** Apply `build_dag()` to every record and organise the results into the two lookup dictionaries.

Iterates over all records, calling `build_dag()` on both the `llm_graph` and `ground_truth_graph` fields. Stores LLM graphs under the `(task_id, model, rep)` key and ground truth graphs under `task_id` alone, deduplicating since the ground truth is the same across all repetitions of a task.

Prints a summary on completion: total records loaded, unique tasks, models, and reps per task.

### 2.4 — Execution cell

**Purpose:** Call the loading pipeline and store the result in `llm_graphs` and `ground_truth_graphs`.

A single short cell that calls `load_raw_records()` then `build_all_graphs()`. Keeping the function definitions separate from the call makes it easy to reload data with a different path without re-running definition cells.

---
## Section 3: Sanity Checks & Data Overview

### Rationale
Graph similarity metrics assume the inputs are valid DAGs. If any LLM output contains a cycle, it is structurally broken and will produce misleading similarity scores — a reversed edge looks like a hallucination rather than a direction error. These checks must run before any metric computation.

This section also surfaces the basic shape of the dataset — size distributions, vocabulary coverage — so that later results can be interpreted in context. For example, knowing that a task's ground truth has 3 nodes vs. 15 nodes is essential context when reading its consistency score.

Any graphs that fail validation are added to an `EXCLUDED` set and skipped in all downstream cells.

### 3.1 — `check_dag_validity(graphs)`

**Purpose:** Run `nx.is_directed_acyclic_graph()` on every LLM graph and return a report of failures.

Returns a dataframe with columns `task_id`, `model`, `rep`, `is_valid`, and `cycle` (the offending cycle if one is found). Prints a summary count of valid vs. invalid graphs.

Graphs that fail are stored in a module-level `EXCLUDED` set — a set of `(task_id, model, rep)` tuples that all later functions will skip. This ensures cycle-containing outputs never silently corrupt similarity calculations.

### 3.2 — Graph size distribution

**Purpose:** Plot histograms of node count and edge count across all LLM graphs and ground truth graphs side by side.

Shows whether LLM outputs are systematically larger or smaller than the ground truth. A heavy right tail (some very large LLM graphs) is an early signal of hallucination-heavy tasks. A heavy left tail suggests the model is under-generating tools.

Also print the min, median, and max node/edge count for both LLM and ground truth distributions.

### 3.3 — Tool vocabulary coverage

**Purpose:** Extract the full set of unique tool names seen across all LLM graphs and compare it to the ground truth vocabulary.

Identifies:
- Tools that appear in LLM graphs but never in any ground truth — candidate hallucination names at the global level
- Tools that appear in ground truth but never in any LLM graph — tools the models collectively never use

Print both sets as sorted lists. This is a global view; per-task tool analysis is in Section 6.

### 3.4 — Coefficient of variation (CV) of graph size per task

**Purpose:** For each task × model group, compute the CV of node count and edge count across repetitions.

CV = std / mean. A high CV means the model cannot decide how complex the plan should be — some runs produce 4-node graphs, others produce 12-node graphs for the same prompt. These high-CV tasks are likely candidates for the bimodal plan distributions detected more precisely in Section 5.

Output as a sortable table. Flag the top 10% highest-CV tasks for attention in later sections.

---
## Section 4: Core Similarity & Error Metric Functions

### Rationale
All metric logic lives here as pure functions with no side effects — no printing, no plotting, no dataframe construction. This section is a library, not an analysis.

The design principle is that every function takes one or two `nx.DiGraph` objects and returns a scalar or dict. This makes them composable: the pairwise matrix function calls the scalar function N² times; the per-task consistency function calls the pairwise matrix function.

Keeping metrics here means any section can call them, and swapping in a new metric later (e.g. WL kernel similarity instead of edge Jaccard) requires changing only this section — nothing downstream breaks.

### 4.1 — `edge_jaccard(G1, G2) → float`

**Purpose:** Compute the Jaccard similarity between the edge sets of two directed graphs.

Formula: `|edges(G1) ∩ edges(G2)| / |edges(G1) ∪ edges(G2)|`

Edges are `(source, target)` tuples — direction matters. Since nodes are named tool strings, edges compare directly with no alignment step. Returns 1.0 if both graphs are empty (both trivially identical), 0.0 if they share no edges at all.

This is the primary similarity signal used throughout the notebook. A high score means the same tool dependencies are wired the same way.

### 4.2 — `node_jaccard(G1, G2) → float`

**Purpose:** Compute Jaccard similarity on node sets only, ignoring edges entirely.

Formula: `|nodes(G1) ∩ nodes(G2)| / |nodes(G1) ∪ nodes(G2)|`

Used as a companion signal to edge Jaccard. When edge Jaccard is low, node Jaccard disambiguates the cause: if node Jaccard is also low, the model is selecting different tools entirely; if node Jaccard is high but edge Jaccard is low, the model has the right tools but wires them differently. These are very different failure modes requiring different interventions.

### 4.3 — `edge_precision_recall_f1(G_pred, G_gt) → dict`

**Purpose:** Compute precision, recall, and F1 of the predicted graph's edges against the ground truth.

- **Precision** = correct edges / all predicted edges → how many of the model's dependency claims are correct
- **Recall** = correct edges / all ground truth edges → how many required dependencies did the model capture
- **F1** = harmonic mean of the two

Unlike Jaccard (which treats both directions of error symmetrically), P/R/F1 reveals *which direction* the model errs in: low precision = hallucinating extra edges; low recall = missing required edges. This asymmetry is critical for model comparison in Section 7.

### 4.4 — `directionality_errors(G1, G2) → int`

**Purpose:** Count the number of edges that exist in both graphs but with direction reversed.

For each edge `(u, v)` in G1, check whether `(v, u)` exists in G2 while `(u, v)` does not. These are cases where the model identified the correct tool pair but got the dependency direction wrong — a qualitatively different error from a missing edge, since the tools are present and connected, just in the wrong order.

Returns a raw count. Callers can normalise by total edge count to produce a rate.

### 4.5 — `critical_path_delta(G1, G2) → int`

**Purpose:** Compute the absolute difference in longest path length between two DAGs.

Uses `nx.dag_longest_path_length()` on both graphs and returns `|len(G1) - len(G2)|`. A delta of 0 means both plans have the same critical path depth even if they differ in other ways. A large delta means the model is producing a fundamentally shallower or deeper dependency chain than the reference.

When comparing against ground truth, this captures whether the model preserved the sequential depth of the task even if individual edges differ.

### 4.6 — `pairwise_similarity_matrix(graphs) → np.ndarray`

**Purpose:** Given a list of `nx.DiGraph` objects, compute the full N×N pairwise edge Jaccard similarity matrix.

The matrix is symmetric with 1.0 on the diagonal. Only the upper triangle is computed; the lower triangle is filled by reflection to avoid redundant computation.

This is the central data structure for all repetition consistency analysis. It is called once per `(task_id, model)` group, passing the K repetition graphs for that group. The resulting matrix is then passed to `variability_stats()`.

### 4.7 — `variability_stats(similarity_matrix) → dict`

**Purpose:** Summarise a pairwise similarity matrix into a set of scalar statistics that characterise the consistency of a group of graphs.

Extracts the upper triangle (unique pairs only), converts to distances (`1 - similarity`), and computes:
- `mean_similarity` — the overall consistency score for this group
- `std_similarity` — spread; low std means the model is reliably consistent or reliably inconsistent
- `min_similarity` — worst-case divergence; the most different pair of plans in the group
- `n_clusters` — number of distinct plan families detected via Ward hierarchical clustering at `CONSISTENCY_CLUSTER_DISTANCE`
- `centroid_idx` — index of the graph with the lowest mean distance to all others; the most representative plan in the group
- `outlier_mask` — boolean array identifying runs whose mean distance to others exceeds mean + 2×std

Returns a dict. The calling loop converts this dict into a row of `consistency_df`.

### 4.8 — `classify_errors(G_pred, G_gt) → dict`

**Purpose:** Decompose the difference between a predicted graph and the ground truth into a structured taxonomy of error types.

Returns counts for each category:
- `missing_nodes` — tools in the ground truth not present in the prediction
- `hallucinated_nodes` — tools in the prediction not present in the ground truth
- `missing_edges` — dependency edges in the ground truth not captured
- `hallucinated_edges` — dependency edges in the prediction not in the ground truth
- `direction_errors` — edges present in both but with direction reversed

Note: missing edges whose endpoints are also missing nodes should be counted separately from missing edges whose endpoints are both present — the latter is a purer wiring error and more informative.

---
## Section 5: Repetition Consistency Analysis

### Rationale
This is the core analysis section. It answers: **for a fixed task and model, how much do the generated plans vary across repeated runs?**

The approach is to group graphs by `(task_id, model)`, compute the pairwise similarity matrix within each group, and extract summary statistics using `variability_stats()`. This produces a flat `consistency_df` dataframe — one row per `(task_id, model)` — that all subsequent sections aggregate from.

Beyond the scalar summary, this section also builds two per-element stability measures: edge frequency (how often each `(u→v)` dependency appears across repetitions) and node frequency (how often each tool appears). These are richer than the scalar stats and support the visualisations in Section 8.

### 5.1 — Build `consistency_df`

**Purpose:** Iterate over all `(task_id, model)` groups, compute the pairwise similarity matrix, and collect `variability_stats()` output into a single dataframe.

For each group:
1. Collect the K repetition graphs, filtering out any whose key appears in `EXCLUDED`
2. Call `pairwise_similarity_matrix()` on them
3. Call `variability_stats()` on the result
4. Append a row to the output with `task_id`, `model`, and all stat fields

The resulting `consistency_df` is the central output of this section. Every later analysis either filters, groups, or pivots it. Print its shape and a grouped summary (mean consistency per model) on completion.

### 5.2 — Edge stability per `(task_id, model)`

**Purpose:** For every `(task_id, model)` group, count how many of the K repetitions contain each edge `(u→v)`. Normalise by K to get a frequency in [0, 1].

An edge with frequency 1.0 is a **certain** dependency — the model always includes it. An edge with frequency 0.1 is **flickering** — it occasionally appears but is not a stable part of the plan.

Store results as a dict mapping `(task_id, model, u, v)` → frequency. This is used in the edge stability bar chart in Section 8 and in the consistent hallucination analysis below.

### 5.3 — Node stability per `(task_id, model)`

**Purpose:** Same as edge stability but for individual tool nodes — how often does each tool appear across the K repetitions?

A node with frequency 1.0 is a stable tool selection; the model always uses it. A node with frequency 0.3 appears in roughly 1 in 3 runs — the model is uncertain whether to include it at all.

Store as a dict mapping `(task_id, model, tool_name)` → frequency. Used to distinguish two failure modes: a low-consistency group where the same tools are rewired differently (high node stability, low edge stability) vs. a group where tool selection itself is uncertain (low node stability). These are different root causes.

### 5.4 — Consistent hallucination detection

**Purpose:** Identify tools that appear in the majority of a task's repetitions but are absent from the ground truth.

For each `(task_id, model)`, cross-reference high-frequency nodes (frequency ≥ 0.7) against the ground truth node set. Any tool that is frequently included but should not be present is a **consistent hallucination** — a systematic model bias rather than stochastic noise.

Consistent hallucinations are more concerning than random noise because they suggest the model has a stable but wrong belief about the task. Output as a table: `task_id`, `model`, `tool_name`, `frequency`.

### 5.5 — Bimodal distribution detection

**Purpose:** Identify tasks where the pairwise similarity distribution is bimodal — the model switches between two distinct plan types rather than producing consistently similar or uniformly random output.

For each `(task_id, model)` group, inspect the `n_clusters` value from `consistency_df`. Groups with `n_clusters == 2` are candidates for bimodal behaviour. For these, retrieve the cluster labels from the Ward clustering and inspect the centroid graph of each cluster to characterise what the two plan families look like — for example, a short sequential plan vs. a longer plan with parallel branches.

Output a list of bimodal tasks with cluster sizes and a brief structural description of each cluster.

---
## Section 6: Error Taxonomy vs. Ground Truth

### Rationale
Section 5 measures consistency *across runs of the same task*. This section measures accuracy *against the ground truth*. The two are complementary: a model can be highly consistent (always produces the same plan) but consistently wrong, or highly accurate on average but wildly inconsistent run to run.

By classifying errors into typed categories, we can understand *why* a model underperforms — not just that it does. This is essential for actionable feedback. The section also identifies structural patterns: which tools are reliable anchors, which are traps, and whether errors tend to cascade downstream once they start.

### 6.1 — Build `error_df`

**Purpose:** Apply `classify_errors()` and `edge_precision_recall_f1()` to every `(task_id, model, rep)` graph against its ground truth, and collect results into a single dataframe.

One row per `(task_id, model, rep)` with columns: `missing_nodes`, `hallucinated_nodes`, `missing_edges`, `hallucinated_edges`, `direction_errors`, `total_errors`, `error_rate` (total errors normalised by ground truth size), `edge_precision`, `edge_recall`, and `edge_f1`.

Keeping all accuracy measures in one dataframe means Section 7 can aggregate them by model in a single groupby. Print a grouped summary (mean edge F1 per model) on completion.

### 6.2 — Error propagation analysis

**Purpose:** For each missing node in a predicted graph, measure how many of its descendant nodes in the ground truth are also missing from the prediction.

Uses `nx.descendants()` on the ground truth graph to find all nodes downstream of each missing node, then counts how many of those descendants are also absent from the prediction.

A high propagation rate means that a single missing tool causes a cascade of downstream failures — the model loses the entire subtree rooted at the missing node. A low rate means missing nodes tend to be leaves or isolated branches with minimal downstream impact.

Output: a per-task mean propagation depth, and identification of the ground truth nodes whose absence causes the most downstream damage. These are the highest-leverage nodes for model improvement.

### 6.3 — Tool reliability: safe anchors & traps

**Purpose:** For every tool in the ground truth vocabulary, compute its correct inclusion rate across all `(task_id, model, rep)` triples where it appears in the ground truth.

- **Safe anchors**: tools with a high correct inclusion rate (e.g. ≥ 0.9) — nearly always correctly placed by all models regardless of task or phrasing
- **Traps**: tools with a high hallucination rate — frequently appearing in LLM graphs for tasks where they are not in the ground truth

Output two ranked tables: top 10 most reliable tools and top 10 most frequently hallucinated tools, with counts and rates across all models. Anchors and traps are properties of the tools themselves, not of any single model.

---
## Section 7: Model Comparison

### Rationale
Sections 5 and 6 produce per-`(task_id, model)` results. This section aggregates those results *across tasks* to produce per-model summaries enabling direct model comparison.

This section contains no new graph-level computation — it only groups, aggregates, and pivots `consistency_df` and `error_df`. If either of those dataframes has not been built, this section will fail.

The key insight this section surfaces is whether models differ primarily in *consistency* (how stable their plans are) or *accuracy* (how close their plans are to ground truth) — and whether those two dimensions actually correlate.

### 7.1 — Per-model consistency leaderboard

**Purpose:** Aggregate `consistency_df` by model to produce a per-model consistency ranking.

For each model, compute the mean and standard deviation of `mean_similarity` across all tasks. A model with high mean and low std is reliably consistent across all task types. A model with high mean but high std is consistent on some tasks but erratic on others — a different failure profile.

Also report the proportion of tasks where `n_clusters > 1` (the model switches between plan modes) for each model. Output as a ranked table, sorted by mean consistency score descending.

### 7.2 — Precision vs. recall tradeoff per model

**Purpose:** Aggregate `error_df` by model to show whether each model tends to over-generate (hallucinate) or under-generate (miss) edges and nodes.

Compute mean edge precision and mean edge recall per model. A model in the upper-right quadrant is accurate overall. A model along the left edge is conservative (high recall, low precision — it includes the right edges but adds extras). A model along the bottom edge is over-selective (high precision, low recall — it only includes edges it is confident about but misses many).

This framing reveals systematic model biases that a single F1 score would obscure. Output as both a table and a scatter plot (built in Section 8).

### 7.3 — Cross-model agreement on failures

**Purpose:** Identify tasks where all models fail — producing low edge F1 regardless of their individual tendencies.

For each task, compute the minimum edge F1 across all models. Tasks where even the best model scores below a threshold (e.g. F1 < 0.3) are **universally hard** — the failure is a property of the task structure, not of any individual model.

Output a ranked list of universally hard tasks. These are prime candidates for qualitative inspection: what structural properties do they share? Long critical paths? Many parallel branches? Rare or ambiguously named tools?

---
## Section 8: Visualisations

### Rationale
All visualisations are collected here. Each plot cell is self-contained — it reads from dataframes and structures already built in sections 5–7 and produces a single figure. Any individual plot can be re-run or modified without re-running the full analysis pipeline.

The visualisations are ordered from high-level summaries (heatmaps, scatter plots) to detailed per-task views (graph overlays, edge stability charts). A reader can scan the high-level views to identify interesting tasks, then drill into the detail views for those specific cases.

All plots should save to a `figures/` directory alongside the notebook as well as displaying inline, so outputs are portable and shareable independently of the notebook.

### 8.1 — Pairwise similarity heatmap

**Purpose:** For a selected `(task_id, model)` group, display the K×K pairwise edge Jaccard similarity matrix as a clustered heatmap.

Use `seaborn.clustermap` with Ward linkage so that similar runs are grouped together. The dendrogram on the axes reveals cluster structure visually — a clean block diagonal pattern means the model has a single stable plan; multiple off-diagonal blocks indicate distinct plan families.

Colour scale runs from 0 (dissimilar) to 1 (identical). Make the target `task_id` and `model` selectable via variables at the top of the cell so this plot can be re-run for any group of interest without touching the rest of the cell.

### 8.2 — MDS scatter plot

**Purpose:** Embed all repetition graphs for a given task into 2D using Multidimensional Scaling on the pairwise distance matrix (`1 - similarity`), and plot as a scatter coloured by model.

MDS preserves pairwise distances in 2D as faithfully as possible, so clusters of points represent groups of structurally similar plans. If all models' runs form a single tight cluster, they broadly agree on the plan structure. If models form separate clusters, each model has a distinct planning style for this task.

Annotate each point with its repetition index. Include a stress value annotation to indicate how well the 2D embedding preserves the true pairwise distances.

### 8.3 — Graph overlay: LLM prediction vs. ground truth

**Purpose:** Draw a single LLM graph side-by-side with its ground truth, using colour to indicate the status of each node and edge.

Node colouring:
- **Green** — present in both ground truth and prediction (correct)
- **Red** — present in prediction but not ground truth (hallucinated)
- **Grey with dashed border** — present in ground truth but missing from prediction

Edge colouring follows the same scheme. Direction errors (reversed edges) should be shown in amber.

Use a hierarchical layout keyed on topological generation so the DAG structure reads top-to-bottom. Default to showing the centroid run (from `variability_stats`) alongside the ground truth. Make `task_id`, `model`, and `rep` selectable at the top of the cell.

### 8.4 — Edge stability bar chart

**Purpose:** For a selected `(task_id, model)`, plot each unique edge `(u→v)` seen across all repetitions on the x-axis, with its appearance frequency on the y-axis.

Colour bars by status relative to the ground truth:
- **Green** — edge is in the ground truth and frequently appears (correct and stable)
- **Amber** — edge is in the ground truth but rarely appears (correct but unstable)
- **Red** — edge is not in the ground truth but frequently appears (stable hallucination)
- **Light red** — edge is not in the ground truth and rarely appears (noisy hallucination)

Sort bars by frequency descending. Draw a horizontal dashed line at the threshold used to define certain vs. flickering edges. This makes the raw edge stability data from Section 5 directly interpretable.

### 8.5 — Consistency vs. accuracy scatter

**Purpose:** Plot each `(task_id, model)` group as a point with mean consistency score (edge Jaccard across reps) on the x-axis and mean edge F1 against ground truth on the y-axis. Colour points by model.

This produces a 2×2 quadrant view of model behaviour:
- **Top-right** — consistent and accurate: the model reliably produces correct plans
- **Top-left** — inconsistent but accurate on average: sometimes right but varies widely
- **Bottom-right** — consistent but wrong: reliably produces the same incorrect plan
- **Bottom-left** — inconsistent and inaccurate: chaotic and incorrect

The bottom-right quadrant is the most analytically interesting: tasks where the model is highly self-consistent but diverges from ground truth suggest a systematic misunderstanding of the task rather than stochastic noise.

### 8.6 — Task difficulty dendrogram

**Purpose:** Cluster tasks by their full difficulty profile vector and display as a dendrogram.

Each task is represented as a feature vector containing: mean edge F1 averaged across models, mean consistency score, ground truth node count, ground truth edge count, and longest path length. Hierarchical clustering with Ward linkage on this vector groups tasks that are jointly hard or easy in the same structural ways.

The dendrogram reveals whether difficulty clusters structurally — for example, whether all multi-server tasks cluster together, or whether long critical path tasks form their own group. Annotate leaf nodes with task identifiers and optionally colour them by a known task metadata field (domain, server count) to test whether structural clusters align with known task categories.